# Mini-TP — Dessiner puis implémenter des graphes LangGraph

Objectif : apprendre à passer d’un **graphe visuel** à un `_build_graph()`, et inversement.

Message pédagogique :

> Un agent LangGraph est une machine à états.
> - Les nœuds sont des étapes de calcul.
> - Les arêtes sont des transitions.
> - Les arêtes conditionnelles sont des décisions.

Ce notebook contient trois exercices :

1. **Graphe donné → écrire `_build_graph()`**
2. **Code `_build_graph()` donné → dessiner le graphe**
3. **Objectif produit donné → concevoir le graphe puis écrire `_build_graph()`**
4. Commencer à réfléchir à un objectif perso pour un projet et en faire le graphe


# Exercice 1 — Graphe donné → écrire `_build_graph()`

Ici, on ne reprend pas la boucle tool-calling du TD précédent.

On veut construire un assistant qui route les demandes utilisateur :

- si la demande est une question simple, on répond directement ;
- si la demande nécessite des documents, on fait d’abord une étape de retrieval ;
- si la demande est hors périmètre, on renvoie une réponse de fallback.

Le graphe est fourni ci-dessous. Votre objectif : écrire le `_build_graph()` correspondant.

![Description de l'image](files/examples_graphs/exercise_1_router_rag.png)

**Exercice:** Complétez le `_build_graph()` suivant.

Aide :

- les nœuds sont :
  - `classify_request`
  - `answer_directly`
  - `retrieve_documents`
  - `answer_with_context`
  - `fallback_answer`
- la fonction de routage conditionnel est `_route_after_classification`
- elle doit renvoyer une des trois chaînes :
  - `"simple"`
  - `"needs_docs"`
  - `"out_of_scope"`

In [ ]:
def _build_graph(self):
    graph = StateGraph(AgentState)

    # TODO: ajouter les nœuds

    # TODO: ajouter l'arête de départ

    # TODO: ajouter les arêtes conditionnelles après classify_request

    # TODO: ajouter les arêtes finales

    return graph.compile()

## Correction possible

In [ ]:
def _build_graph(self):
    graph = StateGraph(AgentState)

    graph.add_node("classify_request", self._classify_request)
    graph.add_node("answer_directly", self._answer_directly)
    graph.add_node("retrieve_documents", self._retrieve_documents)
    graph.add_node("answer_with_context", self._answer_with_context)
    graph.add_node("fallback_answer", self._fallback_answer)

    graph.add_edge(START, "classify_request")

    graph.add_conditional_edges(
        "classify_request",
        self._route_after_classification,
        {
            "simple": "answer_directly",
            "needs_docs": "retrieve_documents",
            "out_of_scope": "fallback_answer",
        },
    )

    graph.add_edge("retrieve_documents", "answer_with_context")
    graph.add_edge("answer_directly", END)
    graph.add_edge("answer_with_context", END)
    graph.add_edge("fallback_answer", END)

    return graph.compile()

### Questions de compréhension

1. Quel nœud produit l’information utilisée pour router ?
2. Quelle fonction décide de la prochaine étape ?
3. Pourquoi `retrieve_documents` ne va-t-il pas directement vers `END` ?
4. Quelle différence entre un nœud et une fonction de routage ?

# Exercice 2 — `_build_graph()` donné → dessiner le graphe

Cette fois, on donne le code, il faut dessiner le graphe sur papier. Le graphe représente un assistant RAG qui vérifie si le contexte récupéré est suffisant.

In [ ]:
def _build_graph(self):
    graph = StateGraph(AgentState)

    graph.add_node("retrieve_documents", self._retrieve_documents)
    graph.add_node("call_model", self._call_model)
    graph.add_node("check_answer", self._check_answer)
    graph.add_node("fallback_answer", self._fallback_answer)

    graph.add_edge(START, "retrieve_documents")
    graph.add_edge("retrieve_documents", "call_model")
    graph.add_edge("call_model", "check_answer")

    graph.add_conditional_edges(
        "check_answer",
        self._route_after_check,
        {
            "good_enough": END,
            "not_enough_context": "fallback_answer",
        },
    )

    graph.add_edge("fallback_answer", END)

    return graph.compile()

## À faire

Dessinez le graphe correspondant sur papier.


# Exercice 3 — Objectif donné → concevoir le graphe puis écrire `_build_graph()`

Objectif produit :

> On veut construire un assistant de recherche.
>
> L’assistant doit :
>
> 1. générer une requête de recherche ;
> 2. chercher des documents ;
> 3. évaluer si le contexte récupéré est suffisant ;
> 4. si le contexte est suffisant, rédiger la réponse ;
> 5. si le contexte est insuffisant, reformuler la requête et chercher à nouveau ;
> 6. s’arrêter après 3 tentatives et produire une réponse de fallback.

Avant d’écrire du code, les élèves doivent dessiner le graphe.

## À faire

1. Identifiez les nœuds.
2. Identifiez les décisions.
3. Identifiez les boucles.
4. Identifiez les conditions d’arrêt.
5. Écrivez `_build_graph()`.

## Correction possible de `_build_graph()`

In [ ]:
def _build_graph(self):
    graph = StateGraph(AgentState)

    graph.add_node("generate_query", self._generate_query)
    graph.add_node("search_documents", self._search_documents)
    graph.add_node("evaluate_context", self._evaluate_context)
    graph.add_node("reformulate_query", self._reformulate_query)
    graph.add_node("write_answer", self._write_answer)
    graph.add_node("fallback_answer", self._fallback_answer)

    graph.add_edge(START, "generate_query")
    graph.add_edge("generate_query", "search_documents")
    graph.add_edge("search_documents", "evaluate_context")

    graph.add_conditional_edges(
        "evaluate_context",
        self._route_after_context_evaluation,
        {
            "sufficient": "write_answer",
            "insufficient": "reformulate_query",
            "too_many_attempts": "fallback_answer",
        },
    )

    graph.add_edge("reformulate_query", "search_documents")
    graph.add_edge("write_answer", END)
    graph.add_edge("fallback_answer", END)

    return graph.compile()

# Exercice bonus — Assistant email avec validation humaine

Objectif :

> Construire un assistant qui rédige un email.
>
> - Il rédige un premier brouillon.
> - Il vérifie si le contenu est sensible.
> - Si ce n’est pas sensible, il termine.
> - Si c’est sensible, il demande une validation humaine.
> - Si l’humain approuve, il termine.
> - Si l’humain rejette, il révise l’email puis redemande une validation.
> - Il s’arrête après 2 rejets.

Les élèves doivent proposer eux-mêmes le graphe et le `_build_graph()`.

## Correction possible du bonus

In [ ]:
def _build_graph(self):
    graph = StateGraph(AgentState)

    graph.add_node("draft_email", self._draft_email)
    graph.add_node("check_sensitivity", self._check_sensitivity)
    graph.add_node("human_review", self._human_review)
    graph.add_node("revise_email", self._revise_email)

    graph.add_edge(START, "draft_email")
    graph.add_edge("draft_email", "check_sensitivity")

    graph.add_conditional_edges(
        "check_sensitivity",
        self._route_after_sensitivity_check,
        {
            "not_sensitive": END,
            "sensitive": "human_review",
        },
    )

    graph.add_conditional_edges(
        "human_review",
        self._route_after_human_review,
        {
            "approved": END,
            "rejected": "revise_email",
            "too_many_rejections": END,
        },
    )

    graph.add_edge("revise_email", "check_sensitivity")

    return graph.compile()

# Conclusion

Avant d’écrire un agent LangGraph, il faut être capable de dessiner son graphe.

Si on ne sait pas dessiner le graphe, on ne sait probablement pas encore clairement ce que l’agent doit faire.